# Deneme 01 · Veri ve çeviri

Atılabilir deneme: kalıcı kod Hafta 2'de `src/ruya/` altına yazılır. Sonuçlar `docs/deneyler/01-veri-ve-ceviri.md` dosyasına işlenir; H1–H4 hipotezleri ve eşikleri orada tanımlıdır.

`# ELLE DOLDUR` işaretli hücreler elle doldurulur; diğer hücreler Colab'de sırayla çalıştırılır.

In [ ]:
# Kurulum hücresi: bütün notebook'larda aynı (bkz. notebooks/README.md)
import os
import subprocess

REPO = "/content/ruya-tabirnamesi"
# os.environ["RUYA_EKLER"] = "colab"  # Chroma, sentence-transformers, Gradio gerekiyorsa
if not os.path.isdir(REPO):
    from google.colab import userdata

    try:
        kimlik = "x-access-token:" + userdata.get("GITHUB_TOKEN") + "@"
    except Exception:
        kimlik = ""  # repo herkese açıksa token gerekmez
    sonuc = subprocess.run(
        ["git", "clone", "-q", f"https://{kimlik}github.com/OgzhnTarhn/ruya-tabirnamesi.git", REPO],
        capture_output=True,
    )
    if sonuc.returncode:
        raise RuntimeError("git clone başarısız: GITHUB_TOKEN'ı ve repo adını kontrol et")
%run {REPO}/notebooks/_kurulum.py

## 0. Kaynakları indir

Nablusi ve Miller Drive'daki `data/raw` klasörüne iner; daha önce indirilmiş dosyalar atlanır.

In [ ]:
!python {REPO}/scripts/download_sources.py --kaynak nablusi miller

from ruya.config import ayarlar

ozet_yolu = ayarlar().ham_klasoru / "KAYNAK_OZETLERI.json"
print(ozet_yolu.read_text(encoding="utf-8"))

## 1. Ayrıştırma işlevleri

Aşağıdaki hücre yalnızca tanım içerir (veri okumaz), bu yüzden Colab dışında da denetlenebilir.

In [ ]:
import re

SAYFA_DESENI = re.compile(r"PageV(\d+)P(\d+)")
SAYFA_SATIRI = re.compile(r"PageV\d+P\d+")
MS_DESENI = re.compile(r"\bms\d+\b")
BASLIK_DESENI = re.compile(r"^\(([^)]*)\)")
BAB_DESENI = re.compile(r"^###\s*\|+\s*")
ALT_ISARETLERI = ("رأى", "رآ", "رؤية")


def nablusi_paragraflari(satirlar: list[str]) -> list[dict]:
    """OpenITI mARkdown metnini paragraflara böler (bkz. docs/02)."""
    basla = 0
    for i, satir in enumerate(satirlar):
        if "#META#Header#End#" in satir:
            basla = i + 1
            break

    paragraflar: list[dict] = []
    bab = None
    acik: list[str] | None = None
    for i in range(basla, len(satirlar)):
        satir = satirlar[i]
        if satir.startswith("### |"):
            bab = BAB_DESENI.sub("", satir).strip()
            acik = None
            continue
        if SAYFA_SATIRI.fullmatch(satir.strip().lstrip("#").strip()):
            continue  # yalnızca sayfa işareti: paragraf başlatmaz
        if satir.startswith("# "):
            acik = [satir]
            paragraflar.append({"bab": bab, "satir": i + 1, "ham": acik})
            continue
        if satir.startswith("~~") and acik is not None:
            acik.append(satir)

    # OpenITI'de işaret sayfayı kapatır: paragraf, kendisinden sonra görülen ilk
    # işaretin sayfasını alır (satır içi işaret de sayılır).
    isaretler = []
    for i in range(basla, len(satirlar)):
        eslesme = SAYFA_DESENI.search(satirlar[i])
        if eslesme:
            isaretler.append((i, f"V{eslesme.group(1)}P{eslesme.group(2)}"))

    sonuc = []
    for no, p in enumerate(paragraflar):
        parcalar = []
        for satir in p["ham"]:
            s = satir[1:] if satir.startswith("#") else satir
            s = s.strip()
            if s.startswith("- "):
                s = s[2:]
            parcalar.append(s)
        metin = " ".join(parcalar)
        metin = metin.replace("~~", " ")
        metin = SAYFA_DESENI.sub(" ", metin)
        metin = MS_DESENI.sub(" ", metin)
        metin = " ".join(metin.split())

        baslik_eslesmesi = BASLIK_DESENI.match(metin)
        baslik = baslik_eslesmesi.group(1) if baslik_eslesmesi else None
        if baslik is None:
            tur = "diger"
        elif any(isaret in baslik for isaret in ALT_ISARETLERI):
            tur = "alt"
        else:
            tur = "baslikli"

        sayfa = next((s for i, s in isaretler if i >= p["satir"] - 1), None)
        sonuc.append(
            {
                "no": no,
                "bab": p["bab"],
                "tur": tur,
                "baslik": baslik,
                "sayfa": sayfa,
                "satir": p["satir"],
                "satir_sayisi": len(p["ham"]),
                "metin": metin,
            }
        )
    return sonuc


def miller_maddeleri(metin: str) -> list[dict]:
    """Gutenberg #926 metnini başlıklı maddelere böler."""
    satirlar = metin.splitlines()
    basla = 0
    for i, satir in enumerate(satirlar):
        if satir.startswith("*** START OF"):
            basla = i + 1
            break
    son = len(satirlar)
    for i in range(basla, len(satirlar)):
        if satirlar[i].startswith("*** END OF"):
            son = i
            break

    maddeler: list[dict] = []
    baslik: str | None = None
    satir_no = 0
    paragraflar: list[str] = []
    tampon: list[str] = []

    def _paragraf_kapat() -> None:
        if tampon:
            paragraflar.append(" ".join(tampon))
            tampon.clear()

    def _madde_kapat() -> None:
        _paragraf_kapat()
        if baslik is not None:
            govde = "\n\n".join(paragraflar)
            govde = re.sub(r"\[\d+\]", "", govde)
            govde = re.sub(r"[ \t]{2,}", " ", govde).strip()
            maddeler.append({"baslik": baslik, "metin": govde, "satir": satir_no})

    for i in range(basla, son):
        satir = satirlar[i]
        eslesme = re.match(r"^_([^_]+)_\.(\[\d+\])?$", satir.strip())
        if eslesme:
            _madde_kapat()
            baslik = eslesme.group(1)
            satir_no = i + 1
            paragraflar = []
            tampon = []
            continue
        if baslik is None:
            continue  # ilk başlıktan önceki satırlar atılır
        if not satir.strip():
            _paragraf_kapat()
        else:
            tampon.append(satir.strip())
    _madde_kapat()
    return maddeler

## 2. H1 · Nablusi: "ح" babından 50 paragraf

Beklenen (docs/02): ~5746 paragraf, ~2400 başlıklı, ~3300 alt, 29 bab.

In [ ]:
import json

import pandas as pd

from ruya.config import ayarlar

ayar = ayarlar()
DENEME = ayar.veri("deneyler", "01")
DENEME.mkdir(parents=True, exist_ok=True)

NABLUSI_SATIRLAR = (ayar.ham_klasoru / "nablusi.txt").read_text(encoding="utf-8").splitlines()
paragraflar = nablusi_paragraflari(NABLUSI_SATIRLAR)

print(f"Toplam paragraf: {len(paragraflar)}")
print("Tür sayıları:")
print(pd.Series([p["tur"] for p in paragraflar]).value_counts().to_string())
bab_sayilari = pd.Series([p["bab"] for p in paragraflar]).value_counts()
print(f"Bab sayısı: {len(bab_sayilari)}")
print("Bab başına paragraf:")
print(bab_sayilari.to_string())

In [ ]:
BAB_ANAHTARI = "الحاء"  # Bab adı farklıysa burayı değiştir

bab_adlari = list(dict.fromkeys(p["bab"] for p in paragraflar if p["bab"]))
bab = next((ad for ad in bab_adlari if BAB_ANAHTARI in ad), None)
if bab is None:
    raise ValueError(f"'{BAB_ANAHTARI}' içeren bab yok. Bablar: {bab_adlari}")

ornek = [p for p in paragraflar if p["bab"] == bab][:50]
(DENEME / "h1_ornek.json").write_text(
    json.dumps(ornek, ensure_ascii=False, indent=1), encoding="utf-8"
)

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 60)
tablo = pd.DataFrame(
    [
        {
            "no": p["no"],
            "tur": p["tur"],
            "baslik": p["baslik"],
            "sayfa": p["sayfa"],
            "satir_sayisi": p["satir_sayisi"],
            "metin": p["metin"][:150],
        }
        for p in ornek
    ]
)
print(f"Bab: {bab} · {len(ornek)} paragraf")
tablo

In [ ]:
def ham_goster(no: int) -> None:
    """Paragrafın ham satırlarını elle denetim için yazdırır."""
    p = paragraflar[no]
    bas = p["satir"] - 1
    print(f"no={no} · bab={p['bab']} · tur={p['tur']} · sayfa={p['sayfa']}")
    for i in range(bas, bas + p["satir_sayisi"]):
        print(f"{i + 1:5}  {NABLUSI_SATIRLAR[i]}")

In [ ]:
# ELLE DOLDUR: sınırı ya da türü yanlış paragrafların numaraları
HATALI_H1: list[int] = []

dogru = len(ornek) - len(HATALI_H1)
print(f"H1: {dogru}/{len(ornek)} doğru (eşik ≥ 48)")

## 3. Miller

Beklenen: ~2256 madde. Örnek, 200–1500 karakterlik maddelerden sabit tohumla seçilir.

In [ ]:
import random

miller_metni = (ayar.ham_klasoru / "miller.txt").read_text(encoding="utf-8")
miller = miller_maddeleri(miller_metni)
print(f"Miller madde sayısı: {len(miller)}")

uygun = [m for m in miller if 200 <= len(m["metin"]) <= 1500]
miller_ornek = random.Random(926).sample(uygun, 10)
for m in miller_ornek:
    print()
    print(f"{m['baslik']} (satır {m['satir']})")
    print(m["metin"][:200])

## 4. Claude çağrısı ve önbellek

Yanıtlar Drive'da `deneyler/01/onbellek` altında saklanır: notebook yeniden çalıştırıldığında API'ye tekrar gidilmez, maliyet doğmaz. Bu bölümde hiçbir rüya metni gönderilmez; yalnızca kaynak metinler ve talimatlar gönderilir.

In [ ]:
import hashlib
import json
import random

import anthropic
from IPython.display import HTML, display

from ruya.config import (
    BATCH_INDIRIMI,
    EFFORT_CEVIRI,
    EFFORT_SEMBOL_ATAMA,
    FIYATLAR,
    MODEL_CEVIRI,
    MODEL_SEMBOL_ATAMA,
    REPO_KOKU,
)

istemci = anthropic.Anthropic()
KULLANIM: list[dict] = []
ONBELLEK = DENEME / "onbellek"
ONBELLEK.mkdir(parents=True, exist_ok=True)


def claude_cagir(ad, model, effort, sistem, kullanici, sema=None, max_tokens=8000) -> dict:
    """Tek bir Claude çağrısı yapar; yanıt Drive'daki önbellekten okunur."""
    anahtar = hashlib.sha256(
        json.dumps(
            {
                "model": model,
                "effort": effort,
                "sistem": sistem,
                "kullanici": kullanici,
                "sema": sema,
                "max_tokens": max_tokens,
            },
            sort_keys=True,
            ensure_ascii=False,
        ).encode("utf-8")
    ).hexdigest()
    onbellek_yolu = ONBELLEK / f"{anahtar}.json"

    if onbellek_yolu.exists():
        kayit = json.loads(onbellek_yolu.read_text(encoding="utf-8"))
        KULLANIM.append({**kayit, "onbellekten": True})
        return kayit

    cikti_ayari: dict = {"effort": effort}
    if sema is not None:
        cikti_ayari["format"] = {"type": "json_schema", "schema": sema}
    yanit = istemci.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=[{"type": "text", "text": sistem, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": kullanici}],
        output_config=cikti_ayari,
    )
    if yanit.stop_reason in ("max_tokens", "refusal"):
        raise RuntimeError(f"Claude çağrısı durdu: {yanit.stop_reason} (ad={ad})")

    metin = next(blok.text for blok in yanit.content if blok.type == "text")
    kayit = {
        "ad": ad,
        "model": model,
        "metin": metin,
        "girdi": yanit.usage.input_tokens,
        "cikti": yanit.usage.output_tokens,
        "onbellek_yazma": yanit.usage.cache_creation_input_tokens or 0,
        "onbellek_okuma": yanit.usage.cache_read_input_tokens or 0,
    }
    onbellek_yolu.write_text(json.dumps(kayit, ensure_ascii=False, indent=1), encoding="utf-8")
    KULLANIM.append({**kayit, "onbellekten": False})
    return kayit


def maliyet(model, girdi, cikti, onbellek_yazma=0, onbellek_okuma=0, batch=False) -> float:
    """Dolar cinsinden maliyet; fiyatlar milyon token başına (ruya.config.FIYATLAR)."""
    fiyat = FIYATLAR[model]
    okuma_fiyati = fiyat.onbellek_okuma if fiyat.onbellek_okuma is not None else fiyat.girdi
    dolar = (
        girdi * fiyat.girdi
        + cikti * fiyat.cikti
        + onbellek_yazma * fiyat.girdi * 1.25
        + onbellek_okuma * okuma_fiyati
    ) / 1_000_000
    return dolar * BATCH_INDIRIMI if batch else dolar

## 5. H2 · Sembol atama (10 alt paragraf)

"ح" babındaki 50 paragraftan 10 alt paragraf seçilir; Claude her birinin hangi sembolü yorumladığına karar verir. Doğru cevap elle işaretlenir (eşik ≥ 8/10).

In [ ]:
alt_ornek = [p for p in ornek if p["tur"] == "alt"][:10]


def onceki_baslik(p: dict) -> str | None:
    """Bu paragraftan önceki en yakın başlıklı paragrafın başlığı (no'ya göre geriye)."""
    for aday in reversed(paragraflar[: p["no"]]):
        if aday["tur"] == "baslikli":
            return aday["baslik"]
    return None


SISTEM_ATAMA = """Aşağıdaki paragraf Nablusi'nin Ta'tîrü'l-enâm adlı eserinden alınmıştır.
Eser, Arapça rüya sembollerini alfabetik sırayla işleyen bir tabir sözlüğüdür.

"ومن رأى" ile başlayan alt paragraflar çoğunlukla kendinden önceki başlıklı maddeyi
sürdürür; ancak bazen yeni bir sembol başlatır. Paragrafın hangi sembolü yorumladığına
karar ver.

Yanıtı yalnızca istenen JSON şemasıyla ver."""

SEMA_ATAMA = {
    "type": "object",
    "additionalProperties": False,
    "required": ["sembol_ar", "sembol_tr", "onceki_basliga_ait", "gerekce", "ozet_tr"],
    "properties": {
        "sembol_ar": {"type": "string"},
        "sembol_tr": {"type": "string"},
        "onceki_basliga_ait": {"type": "boolean"},
        "gerekce": {"type": "string"},
        "ozet_tr": {"type": "string"},
    },
}

atama_sonuclari = []
for p in alt_ornek:
    onceki = onceki_baslik(p)
    kayit = claude_cagir(
        ad="h2",
        model=MODEL_SEMBOL_ATAMA,
        effort=EFFORT_SEMBOL_ATAMA,
        sistem=SISTEM_ATAMA,
        kullanici=f"Önceki başlık: {onceki or '-'}\n\nParagraf: {p['metin']}",
        sema=SEMA_ATAMA,
    )
    atama_sonuclari.append({"no": p["no"], "onceki": onceki, **json.loads(kayit["metin"])})

pd.DataFrame(atama_sonuclari)[
    ["no", "onceki", "sembol_ar", "sembol_tr", "onceki_basliga_ait", "ozet_tr", "gerekce"]
]

In [ ]:
# ELLE DOLDUR: her alt paragrafın ataması doğru mu? (True / False)
DOGRU_H2: list[bool | None] = [None] * 10

if None in DOGRU_H2:
    print(f"H2: {DOGRU_H2.count(None)} hücre hâlâ boş; hepsini doldur.")
else:
    print(f"H2: {sum(DOGRU_H2)}/10 doğru (eşik ≥ 8)")

## 6. H3 · Çeviri (10 Nablusi + 10 Miller)

Puanlama ölçeği: 1 = anlam bozuk, 3 = anlam doğru ama terimler tutarsız, 5 = anlam ve terimler doğru, akıcı. Eşik: ortalama ≥ 3.

In [ ]:
import html

TERIMLER = (REPO_KOKU / "docs" / "terimler.md").read_text(encoding="utf-8").split("\n## 3.")[0]

SISTEM_CEVIRI = f"""Verilen rüya tabiri maddesini Türkçeye çevir.

Kurallar:
- Anlamı koru, hiçbir şey ekleme; süsleme ve yorum katma.
- Kur'an ayeti ve hadis alıntılarını «» içine al.
- Aşağıdaki terim sözlüğündeki karşılıkları tutarlı biçimde kullan.
- Yorum kalıplarını kehanet değil yorum olarak ver:
  "يدل على" → "... ile yorumlanır" / "...ya işaret eder".
- Yalnızca çeviriyi yaz; açıklama ekleme.

Terim sözlüğü:
{TERIMLER}"""

nablusi_ceviri = [p for p in ornek if p["tur"] == "baslikli"][:10]

ceviriler = []
for p in nablusi_ceviri:
    kayit = claude_cagir(
        ad="h3_nablusi",
        model=MODEL_CEVIRI,
        effort=EFFORT_CEVIRI,
        sistem=SISTEM_CEVIRI,
        kullanici=f"Kaynak dil: Arapça\n\n{p['metin']}",
    )
    ceviriler.append(
        {
            "kaynak": "nablusi",
            "no": p["no"],
            "baslik": p["baslik"],
            "asil": p["metin"],
            "ceviri": kayit["metin"],
            "cikti": kayit["cikti"],
        }
    )

for m in miller_ornek:
    kayit = claude_cagir(
        ad="h3_miller",
        model=MODEL_CEVIRI,
        effort=EFFORT_CEVIRI,
        sistem=SISTEM_CEVIRI,
        kullanici=f"Kaynak dil: İngilizce\n\n{m['metin']}",
    )
    ceviriler.append(
        {
            "kaynak": "miller",
            "no": m["satir"],
            "baslik": m["baslik"],
            "asil": m["metin"],
            "ceviri": kayit["metin"],
            "cikti": kayit["cikti"],
        }
    )

(DENEME / "ceviriler.json").write_text(
    json.dumps(ceviriler, ensure_ascii=False, indent=1), encoding="utf-8"
)

satirlar = []
for c in ceviriler:
    yon = "rtl" if c["kaynak"] == "nablusi" else "ltr"
    satirlar.append(
        f"<tr><td dir='{yon}'>{html.escape(c['asil'])}</td><td>{html.escape(c['ceviri'])}</td></tr>"
    )
display(HTML("<table>" + "".join(satirlar) + "</table>"))

In [ ]:
# ELLE DOLDUR: her çevirinin puanı (1–5)
PUAN_NABLUSI: list[int | None] = [None] * 10
PUAN_MILLER: list[int | None] = [None] * 10

for ad, puanlar in (("Nablusi", PUAN_NABLUSI), ("Miller", PUAN_MILLER)):
    dolu = [p for p in puanlar if p is not None]
    if dolu:
        print(f"{ad}: ortalama {sum(dolu) / len(dolu):.2f} ({len(dolu)}/10 dolu, eşik ≥ 3)")
    else:
        print(f"{ad}: henüz puan yok (eşik ≥ 3)")

## 7. H4 · Bütün Nablusi'nin çeviri maliyeti

`count_tokens` ücretsizdir; gerçek token sayılarıyla Batch API maliyeti hesaplanır. Eşik: Opus 5.5 ile ≤ 15 $.

In [ ]:
# Bab başına metin tokenları: her bab tek istekte birleştirilir.
bab_metinleri: dict[str, list[str]] = {}
for p in paragraflar:
    bab_metinleri.setdefault(p["bab"], []).append(p["metin"])

METIN_TOKEN = 0
for metinler in bab_metinleri.values():
    METIN_TOKEN += istemci.messages.count_tokens(
        model=MODEL_CEVIRI,
        messages=[{"role": "user", "content": "\n\n".join(metinler)}],
    ).input_tokens

# İstek başına sabit yük: talimat + kaynak dil satırı.
SISTEM_TOKEN = istemci.messages.count_tokens(
    model=MODEL_CEVIRI,
    system=SISTEM_CEVIRI,
    messages=[{"role": "user", "content": "Kaynak dil: Arapça\n\nx"}],
).input_tokens

# Çıktı/girdi oranı H3'teki 10 Nablusi çağrısından ölçülür. Çıktı, düşünme
# (thinking) tokenlarını da içerir; oran bu yüzden üst sınıra yakındır.
ORAN = sum(c["cikti"] for c in ceviriler if c["kaynak"] == "nablusi") / sum(
    istemci.messages.count_tokens(
        model=MODEL_CEVIRI, messages=[{"role": "user", "content": p["metin"]}]
    ).input_tokens
    for p in nablusi_ceviri
)

N = len(paragraflar)
print(f"Metin tokenı: {METIN_TOKEN:,} · istek başına sistem: {SISTEM_TOKEN:,} · N: {N:,}")
print(f"Çıktı/girdi oranı: {ORAN:.2f}")

# Diğer modellerin tokenizer'ı farklı olabilir; ölçülen ORAN MODEL_CEVIRI'ye aittir.
tahminler = []
for model in FIYATLAR:
    girdi = METIN_TOKEN + N * SISTEM_TOKEN
    cikti = ORAN * METIN_TOKEN
    tahminler.append(
        {
            "model": model,
            "sistem önbelleksiz": maliyet(model, girdi, cikti, batch=True),
            "sistem önbellekli": maliyet(
                model, METIN_TOKEN, cikti, onbellek_okuma=N * SISTEM_TOKEN, batch=True
            ),
        }
    )

H4_TAHMIN = next(t["sistem önbelleksiz"] for t in tahminler if t["model"] == MODEL_CEVIRI)
print("H4 eşiği: MODEL_CEVIRI ile ≤ 15 $")

harcanan = sum(
    maliyet(k["model"], k["girdi"], k["cikti"], k["onbellek_yazma"], k["onbellek_okuma"])
    for k in KULLANIM
    if not k["onbellekten"]
)
print(f"Bu notebook'ta şimdiye kadar harcanan: {harcanan:.2f} $")

pd.DataFrame(tahminler).set_index("model").round(2)

## 8. Sonuç

Aşağıdaki tablo `docs/deneyler/01-veri-ve-ceviri.md` dosyasının "Sonuç" bölümüne yapıştırılır.

In [ ]:
def _ortalama(puanlar: list[int | None]) -> float | None:
    dolu = [p for p in puanlar if p is not None]
    return sum(dolu) / len(dolu) if dolu else None


h1_dogru = len(ornek) - len(HATALI_H1)
h1 = f"{h1_dogru}/{len(ornek)}"
h1_tuttu = "evet" if h1_dogru >= 48 else "hayır"

h2 = "?" if None in DOGRU_H2 else f"{sum(DOGRU_H2)}/10"
h2_tuttu = "?" if None in DOGRU_H2 else ("evet" if sum(DOGRU_H2) >= 8 else "hayır")

h3_nablusi = _ortalama(PUAN_NABLUSI)
h3_miller = _ortalama(PUAN_MILLER)
if h3_nablusi is None or h3_miller is None:
    h3, h3_tuttu = "?", "?"
else:
    h3 = f"Nablusi {h3_nablusi:.2f} · Miller {h3_miller:.2f}"
    h3_tuttu = "evet" if min(h3_nablusi, h3_miller) >= 3 else "hayır"

# H4 tahmini 7. bölümde hesaplanır; o hücre çalışmadıysa "?" yazılır.
h4_tahmin = globals().get("H4_TAHMIN")
h4 = "?" if h4_tahmin is None else f"{h4_tahmin:.2f} $"
h4_tuttu = "?" if h4_tahmin is None else ("evet" if h4_tahmin <= 15 else "hayır")

print("| Hipotez | Ölçülen | Tuttu mu? |")
print("|---|---|---|")
print(f"| H1 | {h1} doğru | {h1_tuttu} |")
print(f"| H2 | {h2} doğru | {h2_tuttu} |")
print(f"| H3 | {h3} | {h3_tuttu} |")
print(f"| H4 | {h4} | {h4_tuttu} |")